# Introduction

This notebook demonstrates the end-to-end process of feature engineering and data prep.

It covers best practices for data modeling, including the creation of primary and foreign key constraints, as well as the use of comments and tags for improved data governance and discoverability. 

The workflow is designed for 16.4LTS or Serverless Databricks environments and leverages modern feature engineering tools to support downstream analytics and machine learning use cases.

This was last updated in October 23rd 2025.

In [0]:
%pip install -q --upgrade faker faker_vehicle databricks-feature-engineering protobuf
%restart_python

### Configuration
TODO: Please read the cell below carefully, and modify it in accordance with your environment/needs.

In [0]:
# If you just installed Faker, you may need to restart the Python kernel for the import to work.
# Use %restart_python or dbutils.library.restartPython() if you see import errors.
from faker import Faker
from faker_vehicle import VehicleProvider
import pandas as pd
import pyspark.sql.functions as F
from databricks.feature_engineering import FeatureEngineeringClient
from databricks.ml_features.entities.online_store import DatabricksOnlineStore

fake = Faker("en-US")
fake.add_provider(VehicleProvider)
num_users = 500 # Number of users to generate

# Location where we'll save our tables
catalog = "cedip"
schema = "features"
user_table_name = f"{catalog}.{schema}.user"
vehicle_table_name = f"{catalog}.{schema}.vehicle"

# Name of the online feature store we'll use/create
online_serving_name = "cedip-online-store" # Must be DNS compliant, only lower case letters, numbers, and hyphens

### Dataset creation
Normally you would already have some data, possibly already enriched or aggregated. In this case, we use Faker to generate the necessary dataset.

In [0]:
# Use Faker to generate data
user_data = []
for _ in range(num_users):
    user = fake.profile()
    car = fake.vehicle_object()
    user.update(car)
    user_data.append(user)
users_df = pd.DataFrame(user_data)

# Create Spark DataFrame from pandas DataFrame
spark_df = spark.createDataFrame(users_df)

# Normalize column names: lowercase, replace spaces and special chars with underscores
def normalize_col(col):
    return (
        col.strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace(".", "_")
        .replace("/", "_")
    )

normalized_cols = [F.col(c).alias(normalize_col(c)) for c in spark_df.columns]
spark_df = spark_df.select(*normalized_cols)

# Identify car columns (assume these exist in the dataset)
car_cols = ['year', 'make', 'model', 'category']
car_cols_norm = [normalize_col(c) for c in car_cols]

# Create vehicle dimension table with car_id hash
vehicle_df = (
    spark_df
    .select(*car_cols_norm)
    .dropDuplicates()
    .withColumn(
        "car_id",
        F.xxhash64(*car_cols_norm)
    )
)

# Join car_id back to main df, drop car columns, and create user_id hash
user_cols = [c for c in spark_df.columns if c not in car_cols_norm]
user_df = (
    spark_df
    .join(vehicle_df, on=car_cols_norm, how="left")
    .drop(*car_cols_norm)
    .withColumn(
        "user_id",
        F.xxhash64(*[F.col(c) for c in user_cols])
    )
)

# Save to Unity Catalog
vehicle_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(vehicle_table_name)
user_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(user_table_name)

### Best practices for tables in UC
When creating assets in UC, follow these guidelines.
- Assign relevant table relation constraints (Primary/Foreign Keys) (for tables only)
- Assign COMMENTS in code. For models/functions/views/volumes you should add comments to the object itself. Tables also should be commented on each column
- Assign TAGS to each asset. Tagging best practices are very subjective, but common tags include:
  - For tables/volumes/functions/views/models: Ownership; Subject; Origin/Project; etc.
  - For columns: DataType/FeatureType; Nullable; Subject;

In [0]:
spark.sql(f"ALTER TABLE {vehicle_table_name} ALTER COLUMN car_id SET NOT NULL;")
spark.sql(f"ALTER TABLE {vehicle_table_name} ADD CONSTRAINT vehicle_pk PRIMARY KEY (car_id);")

spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN user_id SET NOT NULL;")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN car_id SET NOT NULL;")
spark.sql(f"ALTER TABLE {user_table_name} ADD CONSTRAINT user_pk PRIMARY KEY (user_id);")

spark.sql(f"ALTER TABLE {user_table_name} ADD CONSTRAINT user_vehicle_fk FOREIGN KEY (car_id) REFERENCES {vehicle_table_name} (car_id);")

In [0]:
## COMMENTING
# Add table comments
spark.sql(f"COMMENT ON TABLE {vehicle_table_name} IS 'Vehicle dimension table containing unique vehicles with hashed car_id as primary key.'")
spark.sql(f"COMMENT ON TABLE {user_table_name} IS 'User dimension table containing user features and hashed user_id, with foreign key to vehicle.'")

# Add column comments for vehicle table
spark.sql(f"ALTER TABLE {vehicle_table_name} ALTER COLUMN year COMMENT 'Year the vehicle was manufactured.'")
spark.sql(f"ALTER TABLE {vehicle_table_name} ALTER COLUMN make COMMENT 'Automotive brand or manufacturer.'")
spark.sql(f"ALTER TABLE {vehicle_table_name} ALTER COLUMN model COMMENT 'Specific model name or number of the vehicle.'")
spark.sql(f"ALTER TABLE {vehicle_table_name} ALTER COLUMN category COMMENT 'Vehicle classification, e.g., sedan, SUV, truck.'")
spark.sql(f"ALTER TABLE {vehicle_table_name} ALTER COLUMN car_id COMMENT 'Unique hash identifier for the vehicle (primary key).'")

# Add column comments for user table
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN job COMMENT 'User’s current occupation or job title.'")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN company COMMENT 'Organization or employer where the user works.'")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN ssn COMMENT 'User’s Social Security Number (synthetic for demo purposes).'")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN residence COMMENT 'Primary city or region of residence for the user.'")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN current_location COMMENT 'Current geolocation coordinates of the user.'")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN blood_group COMMENT 'User’s blood type, e.g., O+, AB-.'")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN website COMMENT 'List of user’s personal or professional websites.'")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN username COMMENT 'User’s preferred online username or handle.'")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN name COMMENT 'Full name of the user.'")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN sex COMMENT 'User’s gender identity.'")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN address COMMENT 'Full mailing address of the user.'")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN mail COMMENT 'User’s primary email address.'")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN birthdate COMMENT 'Date of birth of the user.'")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN user_id COMMENT 'Unique hash identifier for the user (primary key).'")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN car_id COMMENT 'Foreign key referencing vehicle.car_id.'")

## TAGGING
# Tag columns in vehicle table
spark.sql(f"ALTER TABLE {vehicle_table_name} ALTER COLUMN year SET TAGS ('DataType' = 'Numerical', 'Subject' = 'Vehicle');")
spark.sql(f"ALTER TABLE {vehicle_table_name} ALTER COLUMN make SET TAGS ('DataType' = 'Categorical', 'Subject' = 'Vehicle');")
spark.sql(f"ALTER TABLE {vehicle_table_name} ALTER COLUMN model SET TAGS ('DataType' = 'Categorical', 'Subject' = 'Vehicle');")
spark.sql(f"ALTER TABLE {vehicle_table_name} ALTER COLUMN category SET TAGS ('DataType' = 'Categorical', 'Subject' = 'Vehicle');")
spark.sql(f"ALTER TABLE {vehicle_table_name} ALTER COLUMN car_id SET TAGS ('DataType' = 'ID', 'Subject' = 'Vehicle');")

# Tag columns in user table
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN user_id SET TAGS ('DataType' = 'ID', 'Subject' = 'User');")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN car_id SET TAGS ('DataType' = 'ID', 'Subject' = 'Vehicle');")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN job SET TAGS ('DataType' = 'Categorical', 'Subject' = 'User');")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN company SET TAGS ('DataType' = 'Categorical', 'Subject' = 'User');")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN ssn SET TAGS ('DataType' = 'ID', 'Subject' = 'User');")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN residence SET TAGS ('DataType' = 'Categorical', 'Subject' = 'Location');")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN current_location SET TAGS ('DataType' = 'Categorical', 'Subject' = 'Location');")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN blood_group SET TAGS ('DataType' = 'Categorical', 'Subject' = 'User');")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN website SET TAGS ('DataType' = 'Text', 'Subject' = 'User');")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN username SET TAGS ('DataType' = 'Categorical', 'Subject' = 'User');")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN name SET TAGS ('DataType' = 'Text', 'Subject' = 'User');")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN sex SET TAGS ('DataType' = 'Categorical', 'Subject' = 'User');")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN address SET TAGS ('DataType' = 'Text', 'Subject' = 'Location');")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN mail SET TAGS ('DataType' = 'Text', 'Subject' = 'User');")
spark.sql(f"ALTER TABLE {user_table_name} ALTER COLUMN birthdate SET TAGS ('DataType' = 'Date', 'Subject' = 'User');");

In [0]:
# Docs: https://docs.databricks.com/aws/en/machine-learning/feature-store/online-feature-store#gsc.tab=0
fe = FeatureEngineeringClient()
user_email_address = spark.sql("select current_user() as me").collect()[0]["me"]
fe.set_feature_table_tag(name=vehicle_table_name, key=f"{catalog}FStoreProject", value="feature-store-demo")
fe.set_feature_table_tag(name=vehicle_table_name, key="Owner", value=user_email_address)
fe.set_feature_table_tag(name=user_table_name, key=f"{catalog}FStoreProject", value="feature-store-demo")
fe.set_feature_table_tag(name=user_table_name, key="Owner", value=user_email_address)

## Online Feature Serving
Can be created directly through the FE Client, as an online store. Is powered (and priced) by Lakebase.
#### Pricing
Currently, [Lakebase](https://www.databricks.com/product/pricing/lakebase) charges for the amount of data stored (DSU, Databricks Storage Units) and compute, 1DBU/h for CU_1 (2 for CU_2 and so on).

In [0]:
# Docs: https://docs.databricks.com/aws/en/machine-learning/feature-store/online-feature-store#gsc.tab=0
# Create a new online feature store
try:
  store = fe.get_online_store(name=online_serving_name)
  print(f"Online Feature Store already running: {store.name} with capacity {store.capacity}")
except:
  fe.create_online_store(
      name=online_serving_name,
      capacity="CU_1" # Valid options: "CU_1", "CU_2", "CU_4", "CU_8"
  )

# Enable CDF for both tables
spark.sql(f"ALTER TABLE {vehicle_table_name} SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
spark.sql(f"ALTER TABLE {user_table_name} SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")

# Get the online store instance
online_store = fe.get_online_store(name=online_serving_name)

# Publish vehicle and user tables to the online store with streaming=False. If we set streaming=True, it will create a Streaming SDP to automatically update the online table when the source table is updated. This incurs additional costs.
try:
  pub_result_vehicle = fe.publish_table(
      online_store=online_store,
      source_table_name=vehicle_table_name,
      online_table_name=f"{vehicle_table_name}_online",
      streaming=False
  )
except Exception as e:
  print("Failed to publish vehicle table!")
  print(e, "\n\n")
try:
  pub_result_user = fe.publish_table(
      online_store=online_store,
      source_table_name=user_table_name,
      online_table_name=f"{user_table_name}_online",
      streaming=False
  )
except Exception as e:
  print("Failed to publish vehicle table!")
  print(e, "\n\n")

#### How to use the online features:
[There are many ways](https://docs.databricks.com/aws/en/machine-learning/feature-store/feature-function-serving#gsc.tab=0), for one, if you are using the Feature Engineering Client on your model serving, it automatically uses Online Tables!

Here's one example of how to use it:

In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.is_adult(birthdate DATE)
RETURNS BOOLEAN
LANGUAGE PYTHON
COMMENT 'Returns True if the person is 18 years or older as of today, else False.'
AS $$
from datetime import date

def is_adult(birthdate):
    if birthdate is None:
        return None
    today = date.today()
    age = today.year - birthdate.year - ((today.month, today.day) < (birthdate.month, birthdate.day))
    return age >= 18

return is_adult(birthdate)
$$
""")

In [0]:
from databricks.feature_engineering import (
  FeatureFunction,
  FeatureLookup,
  FeatureEngineeringClient,
)

features = [
  # Lookup columns from the user_online feature table by input user_id
  FeatureLookup(
    table_name=f"{user_table_name}",
    lookup_key="user_id",
    feature_names=[
      "job", "company", "ssn", "residence", "current_location", "blood_group",
      "website", "username", "name", "sex", "address", "mail", "birthdate", "car_id"
    ]
  ),
  # Example: Use is_adult UDF to determine if user is adult based on birthdate
  FeatureFunction(
    udf_name=f"{catalog}.{schema}.is_adult",
    output_name="is_adult",
    input_bindings={"birthdate": "birthdate"},
  ),
]

# Create a FeatureSpec for user features
fe.create_feature_spec(
  name=f"{catalog}.{schema}.user_features_spec",
  features=features,
)

In [0]:
# Delete the serving (no unnecessary spending)
fe.delete_online_store(name=online_serving_name)